# Arkansas traffic assignment benchmark


In [ ]:
import os
import sys
from pathlib import Path

os.environ["AEQ_SHOW_PROGRESS"] = "FALSE"

import numpy as np
import pandas as pd
from aequilibrae import Project
from IPython.display import display
from quetzal.model import stepmodel

if not Path("src/build_graphs.py").is_file():
    sys.path.insert(0, str(Path("benchmarks/arkansas").resolve()))

from src.assignments import CLASSES, assign_period, load_demand
from src.build_graphs import TOLL_FIELDS, build_network

model_path = Path("/mnt/d/models/Arkansas/model")
period = "am"
tolerance = 1e-5
max_iterations = 1
num_threads = 8
method = "bfw"

## Load the network and demand

Both assignments use the same link exclusions, BPR parameters and toll fields. The OMX index supplies the origin and destination IDs.

In [ ]:
with Project(model_path) as project:
    network = build_network(project, [period])
    demand = load_demand(project, period)

pd.Series(
    {
        matrix: demand[segment][matrix].sum()
        for segment, _, _, matrix, _, _, _ in CLASSES
    },
    name="Trips",
)

## AequilibraE assignment

In [ ]:
with Project(model_path) as project:
    aeq_assignment = assign_period(
        project,
        network,
        period,
        demand=demand,
        algorithm=method,
        max_iterations=max_iterations,
        rgap=tolerance,
        cores=num_threads,
    )

display(aeq_assignment.report())
aeq_flows = aeq_assignment.results()
aeq_flows[["AUTO_tot", "COMMTRK_tot", "HTRK_tot", "PCE_tot"]].head()

## Quetzal assignment

Quetzal uses PCE-weighted trips. Split parallel links with an intermediate node so each directed edge has a unique pair of endpoints.

In [ ]:
mapping = network.mapping
quetzal_links = pd.DataFrame(
    {
        "link_id": network.link_ids,
        "link_direction": network.directions,
        "a": mapping.node_ids[mapping.link_tails].astype("int64"),
        "b": mapping.node_ids[mapping.link_heads].astype("int64"),
        "time": network.field_values(f"tt_{period}_10"),
        "capacity": network.field_values(f"{period}_assncap_10"),
        "alpha": network.field_values("alpha"),
        "beta": network.field_values("beta"),
        "length": network.field_values("distance"),
        **{field: network.field_values(field) for field in TOLL_FIELDS},
    }
)
link_modes = network.links.set_index("link_id")["modes"]
quetzal_links["segments"] = (
    link_modes.loc[network.link_ids]
    .map(lambda modes: set(modes.replace("c", "cL")))
    .to_numpy()
)
quetzal_links["vdf"] = "bpr"

parallel = quetzal_links.loc[quetzal_links.duplicated(["a", "b"])].copy()
first_dummy = int(mapping.node_ids.max()) + 1
dummy_nodes = np.arange(first_dummy, first_dummy + len(parallel))
quetzal_links.loc[parallel.index, "b"] = dummy_nodes
second_half = parallel.assign(
    a=dummy_nodes, time=0.0, length=0.0, **dict.fromkeys(TOLL_FIELDS, 0.0)
)
quetzal_links = pd.concat([quetzal_links, second_half], ignore_index=True)
# Quetzal needs each connector to meet a node on a road link.
is_connector = quetzal_links["a"].isin(network.centroids) | quetzal_links["b"].isin(
    network.centroids
)
road_nodes = pd.unique(quetzal_links.loc[~is_connector, ["a", "b"]].to_numpy().ravel())
next_node = first_dummy + len(parallel)
stubs = []
for zone_end, road_end in (("a", "b"), ("b", "a")):
    needs_stub = quetzal_links[zone_end].isin(network.centroids) & ~quetzal_links[
        road_end
    ].isin(road_nodes)
    stub = quetzal_links.loc[needs_stub].assign(
        time=0.0, length=0.0, **dict.fromkeys(TOLL_FIELDS, 0.0)
    )
    stub[road_end] = np.arange(next_node, next_node + len(stub))
    quetzal_links.loc[needs_stub, zone_end] = stub[road_end]
    next_node += len(stub)
    stubs.append(stub)

quetzal_links = pd.concat([quetzal_links, *stubs], ignore_index=True)
quetzal_links.index.name = "index"

is_access = quetzal_links["a"].isin(network.centroids)
is_egress = quetzal_links["b"].isin(network.centroids)
sm = stepmodel.StepModel()
sm.road_links = quetzal_links.loc[~(is_access | is_egress)]
sm.zone_to_road = quetzal_links.loc[is_access | is_egress].copy()
# Quetzal expects "eggress" for links that enter a zone.
sm.zone_to_road["direction"] = np.where(
    is_access.loc[sm.zone_to_road.index], "access", "eggress"
)

In [ ]:
matrices = {
    segment: demand[segment][matrix] for segment, _, _, matrix, _, _, _ in CLASSES
}
zone_ids = demand["c"].index
has_demand = np.logical_or.reduce([matrix > 0 for matrix in matrices.values()])
origins, destinations = np.nonzero(has_demand)
sm.volumes = pd.DataFrame(
    {
        "origin": zone_ids[origins],
        "destination": zone_ids[destinations],
        **{
            segment: matrices[segment][origins, destinations] * pce
            for segment, _, _, _, pce, _, _ in CLASSES
        },
    }
)

sm.step_road_pathfinder(
    method=method,
    maxiters=max_iterations,
    tolerance=tolerance * 100.0,
    segments=[segment for segment, *_ in CLASSES],
    cost_functions={
        segment: f"jam_time + {toll} / {vot}"
        for segment, _, _, _, _, vot, toll in CLASSES
    },
    vdf={"bpr": "time * (1 + alpha * (flow / capacity) ** beta)"},
    num_cores=num_threads,
    return_car_los=False,
    log=True,
)

display(pd.Series(np.asarray(sm.relgap) / 100, name="Quetzal relative gap"))
sm.road_links[["link_id", "link_direction", "flow"]].head()